# [RAGAs를 이용한 데이터셋 생성](https://docs.ragas.io/en/stable/howtos/customizations/testgenerator/_language_adaptation/#generate)

## Load Dataset

In [1]:
import pandas as pd 

df = pd.read_csv("./data/cleaning/nutrition.csv")

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")
df.head(2)

(전체 데이터 수, 전체 컬럼 수): (65, 4)


,video_url,title,clean_caption,title_caption
0,https://www.youtube.com/watch?v=-LqEr-PlCz0&li...,저지방우유먹기,"안녕하세요, 소아청소년과 전문의입니다. 저지방 우유를 권하면 '우리 아이가 비만도 ...","[title] 저지방우유먹기 [caption] 안녕하세요, 소아청소년과 전문의입니다..."
1,https://www.youtube.com/watch?v=v43KzkQWXik,과일먹이기,안녕하세요. 소아청소년과 전문의입니다. 과일과 채소가 건강에 좋다는 것은 누구나 아...,[title] 과일먹이기 [caption] 안녕하세요. 소아청소년과 전문의입니다. ...


## Loader

In [2]:
from langchain_community.document_loaders import CSVLoader

# nutrition.csv -> caption만 로드 진행할 수 있는 객체
loader = CSVLoader(
    file_path="./data/cleaning/nutrition.csv" # 파일명 
    , source_column='video_url' # 소스 컬럼 
    , content_columns=['title_caption'] # 컨텐츠(학습) 컬럼 
    , encoding="utf-8"
)

In [3]:
docs = loader.load() # load()를 이용해서 데이터 로드 진행..

In [4]:
# df.shape -> (데이터의 수(=유튜브 영상의 수), 컬럼의 수)
len(docs), df.shape

(65, (65, 4))

In [5]:
docs[0].metadata['source']

'https://www.youtube.com/watch?v=-LqEr-PlCz0&list=PLC6Msm1YCNw8LtglWOviWSMVd-33S_WHM'

In [6]:
docs[0].page_content

"title_caption: [title] 저지방우유먹기 [caption] 안녕하세요, 소아청소년과 전문의입니다. 저지방 우유를 권하면 '우리 아이가 비만도 아닌데 왜 저지방 우유를 먹여야 하느냐'고 반문하시는 부모님들이 많고, 심지어 '저지방 우유를 먹이면 키가 크지 않을까' 걱정하시는 분들도 있습니다. 우유는 기본적으로 건강에 좋은 음식입니다. 처음에는 모유로 키우되 적어도 2세까지는 모유 수유를 권장하며, 돌이 지나 모유나 분유를 중단하면 일반 우유를 마시게 됩니다. 우유는 담백하고 칼슘이 풍부해 아이에게 유익하며, 보통 하루 한두 컵 정도가 적당합니다. 다만 우유에 든 지방은 포화지방이어서 과다 섭취하면 비만·고혈압·심장병 같은 성인병 위험을 높이므로 지방을 줄인 저지방 우유와 무지방 우유가 개발되었습니다. 두 살 미만의 아이들에게는 지방이 두뇌 발달에 중요하므로 전지(일반) 우유를 권장하고, 두 살 이후에는 저지방이나 무지방 우유로 바꾸는 것이 좋습니다. 저지방 우유를 권하는 주된 이유는 성인병 예방이며, 체중이 많은 경우 저지방 우유가 더 권장됩니다. 또한 저지방이나 무지방 우유는 당뇨병 같은 대사질환이나 일부 암 발생 위험을 줄이는 장점이 있습니다. 인터넷에는 우유가 몸에 나쁘다는 주장도 있지만, 그중에는 전지 우유에 관한 이야기인 경우가 많습니다. 하버드대 공식 홈페이지에도 우유의 장점이 소개되어 있으며, 다만 일반 우유 대신 저지방·무지방 우유를 권한다고 되어 있습니다. 전 세계 많은 전문가들도 대체로 동의합니다. 우유를 논할 때는 반드시 일반 우유와 저지방 우유를 구분해야 하며, 체중이 적은 아이는 저지방 우유를 굳이 먹일 필요가 없고 에너지 보충에 신경 써야 합니다. 저지방 우유 제품은 다양하므로 잘 골라서 먹이시고, 한국에서는 지방 함량 2% 우유도 저지방으로 판매되는 경우가 있지만 의학적으로 말하는 저지방 우유는 지방 함량 1% 이하의 제품을 뜻합니다. 유기농 우유를 마시는 것을 반대하진 않지만, 건강을 생각하신다면 저지방이나 무지방 우

## Model

### [OpenAI API Key](https://platform.openai.com/settings/organization/api-keys)

In [7]:
from dotenv import load_dotenv 

# .env 파일에 있는 데이터를 환경변수에 등록해주는 함수 
load_dotenv()

True

## 학습 데이터셋

### [문장을 생성하는 LLM](https://platform.openai.com/docs/models)

In [8]:
# Langchain LLM 객체를 ragas가 인식할 수 있게 변형
# Wrapper -> 감싸다. 또는 형변환 된다...
from ragas.llms import LangchainLLMWrapper
# Langchain에서 인식할 수 있는 OpenaAI LLM 객체
from langchain_openai import ChatOpenAI

# 문장을 생성하는 LLM -> 질문 & 답변 생성을 목적으로...
generator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini"
))

C:\Users\Playdata\AppData\Local\Temp\ipykernel_18272\211073525.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(


### [Embedding model](https://platform.openai.com/docs/guides/embeddings/embedding-models#embedding-models)

In [9]:
# RAGAs의 최신 OpenAIEmbeddings 사용 (client 필요)
from ragas.embeddings import OpenAIEmbeddings
from openai import OpenAI

# Embedding model -> 문장을 벡터로 변환하는 모델...
generator_embeddings = OpenAIEmbeddings(
    # OpenAI client 생성 (환경변수의 OPENAI_API_KEY를 자동으로 사용)
    client=OpenAI(),
    model="text-embedding-3-small"
)

### 페르소나 정의(중요)
> 어떤 질문/답변 쌍을 만들지 정의함 

In [10]:
from ragas.testset.persona import Persona

personas = [
    Persona(
        name="a pediatric nutrition expert", # 소아과 의사 선생님
        # 일반 대중이 이해하기 쉽게 영양을 설명해주는 의사
        role_description=""""
        You are a pediatric nutrition expert who explains infant and toddler nutrition clearly, accurately, and in an easy-to-understand way for parents and caregivers.
        """,
    ),
    Persona(
        name="a child nutrition specialist", # 소아과 의사 선생님
        # 일반 대중이 이해하기 쉽게 영양을 설명해주는 의사
        role_description=""""
        You are a child nutrition specialist who helps parents understand what and how much infants and toddlers should eat, providing practical, age-appropriate feeding guidance.
        """,
    ),
    Persona(
        name="a pediatrician", # 소아과 의사 선생님
        # 일반 대중이 이해하기 쉽게 영양을 설명해주는 의사
        role_description=""""
        You are a pediatrician with expertise in child nutrition, explaining infant and toddler dietary needs, nutrients, and feeding concerns using evidence-based guidance.
        """,
    ),
]

### 학습용 데이터를 만드는 객체

In [11]:
from ragas.testset import TestsetGenerator

# 학습용 데이터를 만드는 객체
generator = TestsetGenerator(
    llm=generator_llm, embedding_model=generator_embeddings, 
    persona_list=personas
)

NERExtractor: 
> RAG 평가용 테스트셋 생성 시, 문서에 등장하는 핵심 엔티티를 뽑아 질문을 자동 생성하는 클래스입니다.

In [12]:
from ragas.testset.transforms.extractors.llm_based import NERExtractor

transforms = [NERExtractor(llm=generator_llm)]

SingleHopSpecificQuerySynthesizer
> Ragas의 테스트셋 생성 기능에서, 단일 문서 기반(single-hop)으로 “특정하고 구체적인 질문”을 자동 생성하는 Synthesizer 클래스입니다.

In [13]:
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer
)

distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0),
]

for query, _ in distribution:
    prompts = await query.adapt_prompts("korean", llm=generator_llm)
    query.set_prompts(**prompts)

### 학습용 데이터셋 생성

In [14]:
dataset = generator.generate_with_langchain_docs(
    docs[:],
    testset_size=200, # 총 몇개의 질문/답변 쌍을 만들지
    transforms=transforms,
    query_distribution=distribution,
)

Applying NERExtractor:   0%|          | 0/65 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/200 [00:00<?, ?it/s]

## 생성된 데이터셋 확인

In [15]:
eval_dataset = dataset.to_evaluation_dataset()

In [16]:
print("Query:", eval_dataset[0].user_input)
print("Reference:", eval_dataset[0].reference)

Query: 하버드대에서 우유에 대해 뭐라고 했어?
Reference: 하버드대 공식 홈페이지에도 우유의 장점이 소개되어 있으며, 다만 일반 우유 대신 저지방·무지방 우유를 권한다고 되어 있습니다.


In [47]:
# 생성된 테스트셋을 pandas DataFrame으로 변환
eval_df = eval_dataset.to_pandas()
eval_df.shape

(200, 6)

In [48]:
eval_df.head(2)

,user_input,reference_contexts,reference,persona_name,query_style,query_length
0,하버드대에서 우유에 대해 뭐라고 했어?,[title_caption: [title] 저지방우유먹기 [caption] 안녕하세...,"하버드대 공식 홈페이지에도 우유의 장점이 소개되어 있으며, 다만 일반 우유 대신 저...",a pediatrician,MISSPELLED,SHORT
1,한국에서 저지방 우유를 먹이는 이유는 무엇인가요?,[title_caption: [title] 저지방우유먹기 [caption] 안녕하세...,"한국에서 저지방 우유를 권하는 주된 이유는 성인병 예방이며, 체중이 많은 경우 저지...",a pediatrician,MISSPELLED,LONG


> Persona

In [49]:
eval_df['persona_name'].unique()

array(['a pediatrician', 'a child nutrition specialist',
       'a pediatric nutrition expert'], dtype=object)

> Query style

In [50]:
eval_df['query_style'].unique()

array(['MISSPELLED', 'POOR_GRAMMAR', 'WEB_SEARCH_LIKE', 'PERFECT_GRAMMAR'],
      dtype=object)

> Query length

In [51]:
eval_df['query_length'].unique()

array(['SHORT', 'LONG', 'MEDIUM'], dtype=object)

### reference_video_url 추가하기 

In [52]:
eval_df.loc[0,['reference_contexts']].values[0][0].replace("title_caption: ", "")

"[title] 저지방우유먹기 [caption] 안녕하세요, 소아청소년과 전문의입니다. 저지방 우유를 권하면 '우리 아이가 비만도 아닌데 왜 저지방 우유를 먹여야 하느냐'고 반문하시는 부모님들이 많고, 심지어 '저지방 우유를 먹이면 키가 크지 않을까' 걱정하시는 분들도 있습니다. 우유는 기본적으로 건강에 좋은 음식입니다. 처음에는 모유로 키우되 적어도 2세까지는 모유 수유를 권장하며, 돌이 지나 모유나 분유를 중단하면 일반 우유를 마시게 됩니다. 우유는 담백하고 칼슘이 풍부해 아이에게 유익하며, 보통 하루 한두 컵 정도가 적당합니다. 다만 우유에 든 지방은 포화지방이어서 과다 섭취하면 비만·고혈압·심장병 같은 성인병 위험을 높이므로 지방을 줄인 저지방 우유와 무지방 우유가 개발되었습니다. 두 살 미만의 아이들에게는 지방이 두뇌 발달에 중요하므로 전지(일반) 우유를 권장하고, 두 살 이후에는 저지방이나 무지방 우유로 바꾸는 것이 좋습니다. 저지방 우유를 권하는 주된 이유는 성인병 예방이며, 체중이 많은 경우 저지방 우유가 더 권장됩니다. 또한 저지방이나 무지방 우유는 당뇨병 같은 대사질환이나 일부 암 발생 위험을 줄이는 장점이 있습니다. 인터넷에는 우유가 몸에 나쁘다는 주장도 있지만, 그중에는 전지 우유에 관한 이야기인 경우가 많습니다. 하버드대 공식 홈페이지에도 우유의 장점이 소개되어 있으며, 다만 일반 우유 대신 저지방·무지방 우유를 권한다고 되어 있습니다. 전 세계 많은 전문가들도 대체로 동의합니다. 우유를 논할 때는 반드시 일반 우유와 저지방 우유를 구분해야 하며, 체중이 적은 아이는 저지방 우유를 굳이 먹일 필요가 없고 에너지 보충에 신경 써야 합니다. 저지방 우유 제품은 다양하므로 잘 골라서 먹이시고, 한국에서는 지방 함량 2% 우유도 저지방으로 판매되는 경우가 있지만 의학적으로 말하는 저지방 우유는 지방 함량 1% 이하의 제품을 뜻합니다. 유기농 우유를 마시는 것을 반대하진 않지만, 건강을 생각하신다면 저지방이나 무지방 우유를 선택하는 것을 잊지 마

In [53]:
# df[['video_url', 'title_caption']].head(2)
df.loc[0,['title_caption']].values[0]

"[title] 저지방우유먹기 [caption] 안녕하세요, 소아청소년과 전문의입니다. 저지방 우유를 권하면 '우리 아이가 비만도 아닌데 왜 저지방 우유를 먹여야 하느냐'고 반문하시는 부모님들이 많고, 심지어 '저지방 우유를 먹이면 키가 크지 않을까' 걱정하시는 분들도 있습니다. 우유는 기본적으로 건강에 좋은 음식입니다. 처음에는 모유로 키우되 적어도 2세까지는 모유 수유를 권장하며, 돌이 지나 모유나 분유를 중단하면 일반 우유를 마시게 됩니다. 우유는 담백하고 칼슘이 풍부해 아이에게 유익하며, 보통 하루 한두 컵 정도가 적당합니다. 다만 우유에 든 지방은 포화지방이어서 과다 섭취하면 비만·고혈압·심장병 같은 성인병 위험을 높이므로 지방을 줄인 저지방 우유와 무지방 우유가 개발되었습니다. 두 살 미만의 아이들에게는 지방이 두뇌 발달에 중요하므로 전지(일반) 우유를 권장하고, 두 살 이후에는 저지방이나 무지방 우유로 바꾸는 것이 좋습니다. 저지방 우유를 권하는 주된 이유는 성인병 예방이며, 체중이 많은 경우 저지방 우유가 더 권장됩니다. 또한 저지방이나 무지방 우유는 당뇨병 같은 대사질환이나 일부 암 발생 위험을 줄이는 장점이 있습니다. 인터넷에는 우유가 몸에 나쁘다는 주장도 있지만, 그중에는 전지 우유에 관한 이야기인 경우가 많습니다. 하버드대 공식 홈페이지에도 우유의 장점이 소개되어 있으며, 다만 일반 우유 대신 저지방·무지방 우유를 권한다고 되어 있습니다. 전 세계 많은 전문가들도 대체로 동의합니다. 우유를 논할 때는 반드시 일반 우유와 저지방 우유를 구분해야 하며, 체중이 적은 아이는 저지방 우유를 굳이 먹일 필요가 없고 에너지 보충에 신경 써야 합니다. 저지방 우유 제품은 다양하므로 잘 골라서 먹이시고, 한국에서는 지방 함량 2% 우유도 저지방으로 판매되는 경우가 있지만 의학적으로 말하는 저지방 우유는 지방 함량 1% 이하의 제품을 뜻합니다. 유기농 우유를 마시는 것을 반대하진 않지만, 건강을 생각하신다면 저지방이나 무지방 우유를 선택하는 것을 잊지 마

In [54]:
eval_df['reference_title_caption'] = eval_df['reference_contexts'].map(
    lambda x: x[0].replace("title_caption: ", "").strip()
)

eval_df[['reference_title_caption', 'reference_contexts']].head(2)

,reference_title_caption,reference_contexts
0,"[title] 저지방우유먹기 [caption] 안녕하세요, 소아청소년과 전문의입니다...",[title_caption: [title] 저지방우유먹기 [caption] 안녕하세...
1,"[title] 저지방우유먹기 [caption] 안녕하세요, 소아청소년과 전문의입니다...",[title_caption: [title] 저지방우유먹기 [caption] 안녕하세...


In [55]:
eval_df = eval_df.merge(
    right=df[['video_url', 'title_caption']], how='inner',
    left_on="reference_title_caption", right_on="title_caption"
)

In [56]:
eval_df.rename(
    columns={
        "video_url": "reference_video_url"
    }, inplace= True
)

In [57]:
eval_df[['user_input', 'reference', 'reference_video_url', 
    'persona_name', 'query_style', 'query_length']].head()

,user_input,reference,reference_video_url,persona_name,query_style,query_length
0,하버드대에서 우유에 대해 뭐라고 했어?,"하버드대 공식 홈페이지에도 우유의 장점이 소개되어 있으며, 다만 일반 우유 대신 저...",https://www.youtube.com/watch?v=-LqEr-PlCz0&li...,a pediatrician,MISSPELLED,SHORT
1,한국에서 저지방 우유를 먹이는 이유는 무엇인가요?,"한국에서 저지방 우유를 권하는 주된 이유는 성인병 예방이며, 체중이 많은 경우 저지...",https://www.youtube.com/watch?v=-LqEr-PlCz0&li...,a pediatrician,MISSPELLED,LONG
2,저지방우유가 아이들한테 왜 좋은지 설명해줄 수 있어요? 그리고 부모님들이 왜 걱정하...,"저지방 우유는 성인병 예방을 위해 권장되며, 체중이 많은 경우 더욱 좋습니다. 부모...",https://www.youtube.com/watch?v=-LqEr-PlCz0&li...,a pediatrician,POOR_GRAMMAR,LONG
3,우유는 아이한테 왜 중요해요?,"우유는 기본적으로 건강에 좋은 음식으로, 담백하고 칼슘이 풍부해 아이에게 유익합니다...",https://www.youtube.com/watch?v=-LqEr-PlCz0&li...,a child nutrition specialist,MISSPELLED,SHORT
4,비타민이 과일에 왜 중요한가요?,"과일에는 우리 몸에 필요한 다양한 비타민과 섬유질이 풍부하고, 칼륨·마그네슘·항산화...",https://www.youtube.com/watch?v=v43KzkQWXik,a child nutrition specialist,MISSPELLED,SHORT


In [58]:
eval_df[['user_input', 'reference', 'reference_video_url', 
    'persona_name', 'query_style', 'query_length']].tail()

,user_input,reference,reference_video_url,persona_name,query_style,query_length
195,이유식 시작하면 아기 분유 얼마나 먹어야 해?,"이유식은 만 6개월에 시작하면 되고, 하루 수유량은 일반적으로 500~600ml 정...",https://www.youtube.com/watch?v=0J6dh_a0I9I,a child nutrition specialist,POOR_GRAMMAR,SHORT
196,오메가-3 지방산이 포함된 고등어를 이유식에 포함시키는 것이 아기에게 어떤 이점을 ...,고등어는 오메가-3 지방산 등 중요한 영양소가 풍부한 아주 좋은 음식입니다. 아기에...,https://www.youtube.com/watch?v=bMJcxCs5eL4,a child nutrition specialist,WEB_SEARCH_LIKE,MEDIUM
197,"고등어 같은 등푸른 생선이 이유식에 좋다고 하는데, 그 이유는 오메가-3 지방산 때...",등푸른 생선인 고등어는 오메가-3 지방산 등 중요한 영양소가 풍부한 아주 좋은 음식...,https://www.youtube.com/watch?v=bMJcxCs5eL4,a child nutrition specialist,POOR_GRAMMAR,LONG
198,흰살 생선은 이유식에 왜 좋다고 알려져 있나요?,흰살 생선은 이유식 초기나 중기에 무난하게 먹일 수 있는 음식으로 알려져 있습니다....,https://www.youtube.com/watch?v=bMJcxCs5eL4,a pediatric nutrition expert,MISSPELLED,MEDIUM
199,노르웨이산 냉동 고등어는 아기 이유식에 괜찮은가요?,"네, 노르웨이산 냉동 고등어는 수은 농도가 낮은 것으로 알려져 있어 아기 이유식으로...",https://www.youtube.com/watch?v=bMJcxCs5eL4,a pediatrician,POOR_GRAMMAR,MEDIUM


> 중복 데이터 제거 

In [59]:
eval_df = eval_df[['user_input', 'reference', 'reference_video_url', 
    'persona_name', 'query_style', 'query_length']]
print(f"befer: {eval_df.shape}")

eval_df.drop_duplicates(inplace=True, subset=['user_input', 'reference'])
print(f"after: {eval_df.shape}")

befer: (200, 6)
after: (198, 6)


> 결측치 확인

In [60]:
eval_df.isnull().sum()

user_input             0
reference              0
reference_video_url    0
persona_name           0
query_style            0
query_length           0
dtype: int64

In [61]:
eval_df.head(2)

,user_input,reference,reference_video_url,persona_name,query_style,query_length
0,하버드대에서 우유에 대해 뭐라고 했어?,"하버드대 공식 홈페이지에도 우유의 장점이 소개되어 있으며, 다만 일반 우유 대신 저...",https://www.youtube.com/watch?v=-LqEr-PlCz0&li...,a pediatrician,MISSPELLED,SHORT
1,한국에서 저지방 우유를 먹이는 이유는 무엇인가요?,"한국에서 저지방 우유를 권하는 주된 이유는 성인병 예방이며, 체중이 많은 경우 저지...",https://www.youtube.com/watch?v=-LqEr-PlCz0&li...,a pediatrician,MISSPELLED,LONG


### 저장하기 

> 데이터를 랜덤하게 정렬하기 

- `sample(frac=1)` : 전체 행을 랜덤 셔플
- `random_state=42` : 재현 가능하도록 시드 고정 (선택)
- `reset_index(drop=True)` : 기존 index 제거 후 0부터 새로 생성

In [62]:
save_df = eval_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [64]:
save_df.head(2)

,user_input,reference,reference_video_url,persona_name,query_style,query_length
0,체중에 대한 잘못된 정보는 무엇인가요?,"'하루 1,000ml만 넘지 않으면 괜찮다'는 이야기는 잘못된 정보입니다. 실제로 ...",https://www.youtube.com/watch?v=08B0VqzITbU,a pediatric nutrition expert,MISSPELLED,SHORT
1,수유에 대해 아기가 스스로 먹게 하려면 어떻게 해야 하나요?,아기를 먹일 때는 시간을 엄격히 정해 억지로 먹이거나 먹는 양을 지나치게 재지 말고...,https://www.youtube.com/watch?v=QLlu_Zmr9wQ,a pediatrician,MISSPELLED,MEDIUM


> 저장하기 

In [65]:
save_df.to_csv("./data/training/nutrition.csv", index=False, header=True)